# Step B — Pairwise Cross-Correlation Polarity
Re-implements `B_CC_all_15OBS.m` in Python for all 21 stations (Y1).  
**Two-window strategy**: short 0.3 s / long 0.5 s, both starting 0.1 s before P pick.  
Filters: |CC|>0.7, |lag|<0.04 s, inter-window lag diff <0.01 s.  
Saves per-station triplets: `(event_idx_i, event_idx_j, polarity_weight)` → `02-data/B_CC_py/`

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import h5py
from pathlib import Path
from numpy.fft import rfft, irfft
from joblib import Parallel, delayed
import scipy.sparse as sp
import time

# ── Paths ─────────────────────────────────────────────────────
DATA   = Path('/Users/mczhang/Documents/GitHub/FM7/02-data')
OUT_B  = DATA / 'B_CC_py'; OUT_B.mkdir(exist_ok=True)
WAVE_H5 = DATA / 'A_ID' / 'waves_Y2_ge8.h5'
YEAR   = 'Y2'
LOC_FILE = DATA / 'A_all' / 'axial.Y2.mldd.loc.260317'
MAX_DIST_KM = 2.0   # Y2 spatial batching: only compute CC for events within 2 km

# ── Parameters (matching B_CC_all_15OBS.m) ────────────────────
FS         = 200          # Hz
P_IDX      = 64           # sample index of P pick (0.32 s * 200 Hz)
LAG_AHEAD  = 20           # 0.1 s * 200 Hz — samples before P for CC window
WIN_START  = P_IDX - LAG_AHEAD   # = 44
SHORT_LEN  = 60           # 0.3 s short window
LONG_LEN   = 100          # 0.5 s long window
CC_THRESH  = 0.7          # |CC| threshold (same as P.b.ccthreshold)
LAG_THRESH = 0.04         # max |lag| in seconds
LAG_DIFF   = 0.01         # max |lag_long − lag_short| in seconds

print(f'CC window start: sample {WIN_START} = {WIN_START/FS:.3f} s before P pick')
print(f'Short window: samples {WIN_START}:{WIN_START+SHORT_LEN}  ({SHORT_LEN/FS*1000:.0f} ms)')
print(f'Long  window: samples {WIN_START}:{WIN_START+LONG_LEN}  ({LONG_LEN/FS*1000:.0f} ms)')


In [ ]:
# ── FFT-based batch CC ────────────────────────────────────────
def _cc_row(i, F_short, F_long, norms_short, norms_long, N_short, N_long, n_events):
    """
    Compute CC of row i against all rows j > i.
    Returns list of (i, j, polarity_weight_long) passing all filters.
    """
    j_range = np.arange(i + 1, n_events)
    if len(j_range) == 0:
        return []

    def batch_cc(Fi, Fbatch, norm_i, norms_j, N_fft, n_samp):
        cc_raw = irfft(Fi[None, :] * np.conj(Fbatch), n=N_fft, axis=1)  # (m, N_fft)
        denom  = (norm_i * norms_j + 1e-12)[:, None]
        cc     = cc_raw / denom
        # Lag index mapping: k < n_samp → lag k/FS; k >= N_fft-n_samp+1 → lag (k-N_fft)/FS
        max_idx = cc.argmax(axis=1)
        min_idx = cc.argmin(axis=1)
        lags_max = np.where(max_idx < n_samp, max_idx, max_idx - N_fft) / FS
        lags_min = np.where(min_idx < n_samp, min_idx, min_idx - N_fft) / FS
        return cc[np.arange(len(j_range)), max_idx], lags_max, \
               cc[np.arange(len(j_range)), min_idx], lags_min

    N_fft_s = 2 ** int(np.ceil(np.log2(2 * N_short - 1)))
    N_fft_l = 2 ** int(np.ceil(np.log2(2 * N_long  - 1)))

    max_s, lag_s_max, min_s, lag_s_min = batch_cc(
        F_short[i], F_short[j_range], norms_short[i], norms_short[j_range],
        N_fft_s, N_short)
    max_l, lag_l_max, min_l, lag_l_min = batch_cc(
        F_long[i],  F_long[j_range],  norms_long[i],  norms_long[j_range],
        N_fft_l, N_long)

    # Dominant CC = whichever of max/min has larger absolute value
    dom_s   = np.where(np.abs(max_s) >= np.abs(min_s), max_s, min_s)
    lag_s   = np.where(np.abs(max_s) >= np.abs(min_s), lag_s_max, lag_s_min)
    dom_l   = np.where(np.abs(max_l) >= np.abs(min_l), max_l, min_l)
    lag_l   = np.where(np.abs(max_l) >= np.abs(min_l), lag_l_max, lag_l_min)

    # Polarity weight = sign(dominant) × |abs(max) − abs(min)|  (col 9 in MATLAB)
    pw_l = np.sign(dom_l) * np.abs(np.abs(max_l) - np.abs(min_l))

    # ── Apply all filters ────────────────────────────────────
    keep = (
        (np.abs(dom_s) >= CC_THRESH) &
        (np.abs(dom_l) >= CC_THRESH) &
        (np.abs(lag_s) < LAG_THRESH) &
        (np.abs(lag_l) < LAG_THRESH) &
        (np.abs(lag_l - lag_s) < LAG_DIFF)
    )
    if not keep.any():
        return []
    return list(zip(
        [i] * keep.sum(),
        j_range[keep].tolist(),
        pw_l[keep].tolist()
    ))


In [ ]:
# ── Load spatial locations for Y2 batching ───────────────────
from scipy.spatial import cKDTree

def load_locs(path):
    """Parse MLDD .loc file → {eid: (lat, lon)} dict."""
    locs = {}
    with open(path) as fh:
        for line in fh:
            p = line.split()
            if len(p) < 11: continue
            try:
                lat, lon = float(p[6]), float(p[7])
                eid = int(p[10])
                locs[eid] = (lat, lon)
            except (ValueError, IndexError):
                continue
    return locs

def latlon_to_km(lats, lons, lat0, lon0):
    """Approx Cartesian km offset from reference point."""
    dlat = np.array(lats) - lat0
    dlon = np.array(lons) - lon0
    x = dlon * np.cos(np.radians(lat0)) * 111.32
    y = dlat * 111.32
    return x, y

eid_locs = load_locs(LOC_FILE)
print(f'Loaded {len(eid_locs):,} event locations from {LOC_FILE.name}')
lat0 = np.mean([v[0] for v in eid_locs.values()])
lon0 = np.mean([v[1] for v in eid_locs.values()])

# ── Main loop: process all stations ──────────────────────────
with h5py.File(WAVE_H5) as f:
    ALL_STAS = sorted(f['waveforms'].keys())
    EID_ALL  = f['event_ids'][:]

print(f'Stations: {ALL_STAS}')
print(f'Total events in H5: {len(EID_ALL):,}')
print(f'Spatial filter: MAX_DIST_KM = {MAX_DIST_KM} km\n')

n_jobs = -1   # use all cores

for sta in ALL_STAS:
    t0 = time.time()
    with h5py.File(WAVE_H5) as f:
        waves = f[f'waveforms/{sta}'][:]   # (N_all, 264)

    # Keep only rows with valid (finite) data
    valid  = np.isfinite(waves).all(axis=1)
    idx_v  = np.where(valid)[0]
    eids_v = EID_ALL[idx_v]
    n      = len(idx_v)
    if n < 10:
        print(f'{sta}: only {n} valid waveforms — skip')
        continue

    # Extract and normalize CC windows
    def _prep(w, start, length):
        seg = w[:, start:start + length].copy()
        norm = np.linalg.norm(seg, axis=1, keepdims=True)
        norm = np.where(norm < 1e-10, 1.0, norm)
        return seg / norm

    sw = _prep(waves[idx_v], WIN_START, SHORT_LEN)
    lw = _prep(waves[idx_v], WIN_START, LONG_LEN)

    # Pre-compute FFTs
    N_fft_s = 2 ** int(np.ceil(np.log2(2 * SHORT_LEN - 1)))
    N_fft_l = 2 ** int(np.ceil(np.log2(2 * LONG_LEN  - 1)))
    F_s = rfft(sw, n=N_fft_s, axis=1)
    F_l = rfft(lw, n=N_fft_l, axis=1)
    # Norms already 1.0 after normalization — but keep for safety
    ns  = np.linalg.norm(sw, axis=1)
    nl  = np.linalg.norm(lw, axis=1)

    # Parallel over rows i
    raw = Parallel(n_jobs=n_jobs, prefer='threads')(
        delayed(_cc_row)(i, F_s, F_l, ns, nl, SHORT_LEN, LONG_LEN, n)
        for i in range(n - 1)
    )

    # Flatten
    triplets = [t for row in raw for t in row]
    if len(triplets) == 0:
        print(f'{sta}: no pairs passed filters')
        continue

    arr = np.array(triplets)   # (M, 3): local_idx_i, local_idx_j, polarity_weight
    # Convert local indices → event_ids
    arr_out = np.column_stack([
        eids_v[arr[:, 0].astype(int)],
        eids_v[arr[:, 1].astype(int)],
        arr[:, 2]
    ])

    np.save(OUT_B / f'B_{YEAR}_{sta}.npy', arr_out)
    print(f'{sta}: {n} events, {len(arr_out):,} pairs  ({time.time()-t0:.1f}s)')

print('\nStep B done.')
